# ROBO CONDUCTOR：文字で指示して演奏する

日本語の文字を入力し、AIが意味を解釈した結果を使ってロボットが演奏します。ノートブック1つで実行でき、追加のZIPは不要です。

最初に「赤、青、緑、赤の順番で叩いて」を実行して動画を確認し、その後に「きらきら星を弾いて」を実行します。各指示を入力した時点で**OpenAI APIに文章が送られ、料金が発生する場合があります**。実行にはAPIキーが必要です。

- AIは3色の順番、または登録曲「きらきら星」を選択します。未知の曲や対応外の指示は実行しません。
- 曲の譜面はあらかじめ用意したもので、AIが作曲するわけではありません。
- 赤＝ド（C4）、青＝ラ（A4）、緑＝ファ（F4）。画面上のパッドの色と対応しています。
- **各指示は別のシミュレーション**で、ロボットの位置は初期状態に戻り、別々の動画になります。AIは各指示を個別に解釈します。
- 物理計算と画像生成はCPUを使用します。GPUは不要です。フルHDは時間がかかるため、最初は確認用画質を推奨します。
- Colab実環境と新規API呼び出しは未検証です。


## 1. ライブラリを準備

Apple Siliconを含む環境で使われる `pybullet-arm64` は、公式PyBulletとは別に保守される互換フォークです。Linux版も使用します。

In [ ]:
import importlib.metadata as metadata
import importlib.util
import platform
import subprocess
import sys

print("Python:", sys.version.split()[0], "OS:", platform.system(), platform.machine())
try:
    upstream_version = metadata.version("pybullet")
except metadata.PackageNotFoundError:
    upstream_version = None
try:
    fork_version = metadata.version("pybullet-arm64")
except metadata.PackageNotFoundError:
    fork_version = None
if upstream_version and fork_version != "3.2.8":
    raise RuntimeError("別のpybulletが入っています。競合を避けるため、新しいランタイムで実行してください。")
required = ("numpy", "imageio", "imageio_ffmpeg", "PIL", "pybullet", "openai")
if fork_version != "3.2.8" or not all(importlib.util.find_spec(name) for name in required):
    packages = ["pybullet-arm64==3.2.8", "numpy>=1.24,<3", "imageio>=2.31,<3", "imageio-ffmpeg>=0.4,<1", "Pillow>=9,<14", "openai>=1,<3"]
    try:
        subprocess.run([sys.executable, "-m", "pip", "install", "--only-binary=:all:", *packages], check=True)
    except subprocess.CalledProcessError:
        raise RuntimeError("ライブラリをインストールできません。Colabの標準ランタイムとネットワークを確認してください。") from None
else:
    print("必要なライブラリを確認しました。")


## 2. 作業フォルダとプログラムを用意

In [ ]:
from pathlib import Path
import os, sys, tempfile

# 実行するたび新しいフォルダを作り、前の結果を上書きしません。
実行フォルダ = Path(tempfile.mkdtemp(prefix="robo_dialogue_", dir=str(Path.cwd())))
os.chdir(実行フォルダ)
if str(実行フォルダ) not in sys.path:
    sys.path.insert(0, str(実行フォルダ))
print("動画などの保存先:", 実行フォルダ)


In [ ]:
%%writefile melody.py
"""Eight-pad contact-driven melody, independent of the original three-pad demo.

Twinkle Twinkle Little Star is the traditional melody arranged here as six
phrases. Its rhythm is one beat per note, two beats at each phrase end.
"""
from __future__ import annotations
import argparse
import json
import math
import subprocess
import tempfile
import wave
from pathlib import Path

import imageio.v2 as imageio
import imageio_ffmpeg
import numpy as np
import pybullet as p
import pybullet_data
from PIL import Image, ImageDraw, ImageFont

HZ = 240
NOTES = ('C4', 'D4', 'E4', 'F4', 'G4', 'A4', 'B4', 'C5')
SOLFEGE = dict(zip(NOTES, ('Do', 'Re', 'Mi', 'Fa', 'Sol', 'La', 'Si', 'Do+')))
FREQ = dict(zip(NOTES, (261.625565, 293.664768, 329.627557, 349.228231,
                        391.995436, 440.0, 493.883301, 523.251131)))
COLORS = dict(zip(NOTES, ((0.85,0.08,0.08,1), (0.95,0.35,0.05,1),
                           (0.88,0.75,0.04,1), (0.15,0.68,0.12,1),
                           (0.02,0.58,0.7,1), (0.1,0.25,0.85,1),
                           (0.55,0.14,0.77,1), (0.8,0.18,0.48,1))))
# Two rows of four. Centers are 15 cm apart; pads are 11 cm wide.
POSITIONS = {n:(x,y,0.055) for n,(x,y) in zip(NOTES,
    ((0.42,-0.225),(0.42,-0.075),(0.42,0.075),(0.42,0.225),
     (0.57,-0.225),(0.57,-0.075),(0.57,0.075),(0.57,0.225)))}
PHRASES = (
    ('C4','C4','G4','G4','A4','A4','G4'),
    ('F4','F4','E4','E4','D4','D4','C4'),
    ('G4','G4','F4','F4','E4','E4','D4'),
    ('G4','G4','F4','F4','E4','E4','D4'),
    ('C4','C4','G4','G4','A4','A4','G4'),
    ('F4','F4','E4','E4','D4','D4','C4'),
)


def make_score(piece):
    if piece == 'scale':
        return [{'note': note, 'beats': 1, 'phrase': 'scale'} for note in NOTES]
    if piece != 'twinkle':
        raise ValueError('Unknown piece')
    return [{'note': note, 'beats': 2 if i == len(phrase)-1 else 1,
             'phrase': phrase_index+1}
            for phrase_index, phrase in enumerate(PHRASES)
            for i,note in enumerate(phrase)]


def synthesize(contacts, duration, beat_seconds, path, rate=48000):
    samples = np.zeros(math.ceil(duration * rate), dtype=np.float64)
    for event in contacts:
        # Wrong-pad contacts also make the sound of the actual pad.
        beats = event.get('beats', 1)
        length = int(min(beat_seconds * beats * 0.88, duration) * rate)
        start = round(event['time_s'] * rate)
        end = min(len(samples), start + length)
        length = end-start
        if length <= 0:
            continue
        t = np.arange(length) / rate
        env = np.minimum(1, t / 0.012) * np.minimum(1, (length/rate-t) / 0.10)
        env *= np.exp(-t*0.5)
        f = FREQ[event['note']]
        tone = (np.sin(2*np.pi*f*t) + 0.22*np.sin(2*np.pi*2*f*t) +
                0.07*np.sin(2*np.pi*3*f*t)) / 1.29
        samples[start:end] += 0.30 * env * tone
    pcm = (np.clip(samples, -0.95, 0.95) * 32767).astype('<i2')
    with wave.open(str(path), 'wb') as f:
        f.setnchannels(1); f.setsampwidth(2); f.setframerate(rate)
        f.writeframes(pcm.tobytes())


def _font(size):
    for path in ('/System/Library/Fonts/Helvetica.ttc',
                 '/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf'):
        try: return ImageFont.truetype(path,size)
        except OSError: pass
    return ImageFont.load_default()


def render(width,height,client,active,piece):
    view=p.computeViewMatrix(cameraEyePosition=[1.18,-1.1,1.25],
                             cameraTargetPosition=[0.34,0,0.28],cameraUpVector=[0,0,1])
    projection=p.computeProjectionMatrixFOV(fov=55,aspect=width/height,nearVal=.05,farVal=5)
    rgba=p.getCameraImage(width,height,viewMatrix=view,projectionMatrix=projection,
                          renderer=p.ER_TINY_RENDERER,physicsClientId=client)[2]
    img=Image.fromarray(np.asarray(rgba,dtype=np.uint8).reshape(height,width,4)[:,:,:3])
    draw=ImageDraw.Draw(img,'RGBA'); sz=max(10,round(height*.034)); font=_font(sz)
    title=_font(max(12,round(height*.045))); margin=max(8,round(height*.025))
    draw.text((margin,margin), 'TWINKLE TWINKLE LITTLE STAR' if piece=='twinkle' else 'DO RE MI - PAD CHECK', font=title, fill=(30,45,65,255))
    y=margin+round(height*.065); line=max(sz+3,round(height*.042))
    for note in NOTES:
        rgb=tuple(round(x*255) for x in COLORS[note][:3]); selected=active == note
        draw.rectangle((margin,y+2,margin+line//2,y+line//2+2),fill=(*rgb,255),outline=(20,20,20,255))
        draw.text((margin+line*.7,y),f'{SOLFEGE[note]}  {note}'+('  *' if selected else ''),font=font,fill=(15,20,30,255))
        y+=line
    return np.asarray(img)


def run(score, piece, output, log, audio, fps=30, width=640, height=360):
    if fps <= 0 or HZ % fps or width < 64 or height < 64 or width%2 or height%2:
        raise ValueError('fps must divide 240; width/height must be even and >=64')
    output, log, audio = map(Path,(output,log,audio))
    if output.suffix.lower() != '.mp4' or audio.suffix.lower() != '.wav':
        raise ValueError('output must be MP4 and audio WAV')
    if len({output.resolve(),log.resolve(),audio.resolve()}) != 3:
        raise ValueError('output, audio and log must be distinct')
    for path in (output,log,audio): path.parent.mkdir(parents=True,exist_ok=True)
    if any(e.get('note') not in NOTES or e.get('beats') not in (1,2) for e in score):
        raise ValueError('Invalid score')
    # Four phase durations add to 1.6 seconds at 30 fps. All are rounded to
    # whole frames, keeping simulation and video clocks aligned for any fps.
    phase_seconds=(.6,.4,.2,.4)
    phase_frames=[max(1,round(s*fps)) for s in phase_seconds]
    beat_frames=sum(phase_frames); beat_seconds=beat_frames/fps
    steps_per_frame=HZ//fps
    client=p.connect(p.DIRECT)
    if client < 0: raise RuntimeError('Could not connect to PyBullet')
    writer=None; frames=0; steps=0; current=None; strikes=[]; contacts=[]
    touching={n:False for n in NOTES}; glow_until={n:-1 for n in NOTES}
    try:
        p.setAdditionalSearchPath(pybullet_data.getDataPath(),physicsClientId=client)
        p.setTimeStep(1/HZ,physicsClientId=client);p.setGravity(0,0,-9.81,physicsClientId=client)
        p.loadURDF('plane.urdf',physicsClientId=client)
        robot=p.loadURDF('kuka_iiwa/model.urdf',[0,0,0],useFixedBase=True,physicsClientId=client)
        col=p.createCollisionShape(p.GEOM_BOX,halfExtents=[.055,.055,.012],physicsClientId=client)
        pads={}
        for note in NOTES:
            vis=p.createVisualShape(p.GEOM_BOX,halfExtents=[.055,.055,.012],rgbaColor=COLORS[note],physicsClientId=client)
            pads[note]=p.createMultiBody(baseMass=0,baseCollisionShapeIndex=col,baseVisualShapeIndex=vis,
                                         basePosition=POSITIONS[note],physicsClientId=client)
        joints=[i for i in range(p.getNumJoints(robot,physicsClientId=client))
                if p.getJointInfo(robot,i,physicsClientId=client)[2]==p.JOINT_REVOLUTE]
        if len(joints)!=7: raise RuntimeError('Unexpected KUKA joint layout')
        ee=6; home=[.36,0,.65]
        with tempfile.TemporaryDirectory(prefix='robo-melody-') as tmp:
            silent=str(Path(tmp)/'silent.mp4')
            writer=imageio.get_writer(silent,fps=fps,codec='libx264',quality=7,macro_block_size=1)
            def move(goal,nframes):
                nonlocal frames,steps,current
                count=nframes*steps_per_frame
                start=np.array(p.getLinkState(robot,ee,computeForwardKinematics=True,physicsClientId=client)[4]);dest=np.array(goal)
                for n in range(count):
                    if steps%steps_per_frame==0:
                        for note,body in pads.items():
                            rgba=COLORS[note]
                            if steps < glow_until[note]: rgba=tuple(.65+.35*v for v in rgba[:3])+(1,)
                            p.changeVisualShape(body,-1,rgbaColor=rgba,physicsClientId=client)
                        active=(current['note'] if current and current['contact_confirmed'] and
                                steps < glow_until[current['note']] else None)
                        writer.append_data(render(width,height,client,active,piece));frames+=1
                    alpha=(n+1)/count; smooth=alpha*alpha*(3-2*alpha)
                    target=(start*(1-smooth)+dest*smooth).tolist()
                    qs=p.calculateInverseKinematics(robot,ee,target,maxNumIterations=100,residualThreshold=1e-4,physicsClientId=client)
                    for j,q in zip(joints,qs):
                        p.setJointMotorControl2(robot,j,p.POSITION_CONTROL,targetPosition=float(q),force=250,physicsClientId=client)
                    p.stepSimulation(physicsClientId=client);steps+=1
                    for note,body in pads.items():
                        points=p.getContactPoints(bodyA=robot,bodyB=body,linkIndexA=ee,physicsClientId=client)
                        points=[pt for pt in points if pt[9]>0]
                        now=bool(points)
                        if now and not touching[note]:
                            pt=max(points,key=lambda v:v[9])
                            event={'note':note,'time_s':steps/HZ,'robot_link':ee,'position_on_pad':list(pt[6]),
                                   'normal_force_n':float(pt[9]),'strike':current['strike'] if current else None,
                                   'intended_note':current['note'] if current else None,
                                   'beats':current['beats'] if current and current['note']==note else 1}
                            contacts.append(event);glow_until[note]=steps+round(.25*HZ)
                            if current and current['note']==note and not current['contact_confirmed']:
                                current.update(contact_confirmed=True,contact_time_s=steps/HZ,
                                               normal_force_n=event['normal_force_n'],position_on_pad=event['position_on_pad'])
                        touching[note]=now
            move(home,round(1.2*fps))
            for i,item in enumerate(score,1):
                note=item['note'];x,y,z=POSITIONS[note]
                current={'strike':i,'note':note,'beats':item['beats'],'phrase':item.get('phrase'),
                         'start_time_s':steps/HZ,'contact_confirmed':False,'contact_time_s':None,'normal_force_n':None}
                strikes.append(current)
                above=[x,y,z+.25]; hit=[x,y,z+.02]
                move(above,phase_frames[0]);move(hit,phase_frames[1]);move(hit,phase_frames[2]);move(above,phase_frames[3])
                for _ in range(item['beats']-1): move(above,beat_frames)
                current['end_time_s']=steps/HZ;current=None
            move(home,round(.6*fps))
            writer.close();writer=None
            if frames*steps_per_frame != steps: raise RuntimeError('video and physics clocks differ')
            duration=frames/fps
            synthesize(contacts,duration,beat_seconds,audio)
            subprocess.run([imageio_ffmpeg.get_ffmpeg_exe(),'-y','-loglevel','error','-i',silent,'-i',str(audio),
                            '-c:v','copy','-c:a','aac','-b:a','192k','-movflags','+faststart','-shortest',str(output)],check=True)
        data={'piece':piece,'source':'hand-authored traditional melody' if piece=='twinkle' else 'scale check',
              'video':str(output),'audio':str(audio),'fps':fps,'width':width,'height':height,'duration_s':duration,
              'frame_count':frames,'simulation_hz':HZ,'nominal_beat_s':beat_seconds,'nominal_bpm':60/beat_seconds,
              'events':strikes,'contacts':contacts,'successful_strikes':sum(e['contact_confirmed'] for e in strikes),
              'total_strikes':len(strikes),
              'notes':['Sound is synthesized from actual force-bearing contact onsets, including unintended pads.',
                       'Pad glow is a color change; audio is not a physical impact sound.',
                       'Beat timing is nominal; actual contact onsets can differ due to robot movement.']}
        log.write_text(json.dumps(data,ensure_ascii=False,indent=2),encoding='utf-8')
        return data
    finally:
        if writer: writer.close()
        p.disconnect(physicsClientId=client)


def main():
    parser=argparse.ArgumentParser(description=__doc__)
    parser.add_argument('--piece',choices=('scale','twinkle'),default='twinkle')
    parser.add_argument('--output',default='output/twinkle.mp4');parser.add_argument('--log',default='output/twinkle.json')
    parser.add_argument('--audio',default='output/twinkle.wav');parser.add_argument('--fps',type=int,default=30)
    parser.add_argument('--width',type=int,default=640);parser.add_argument('--height',type=int,default=360)
    args=parser.parse_args()
    r=run(make_score(args.piece),args.piece,args.output,args.log,args.audio,args.fps,args.width,args.height)
    print(f"{r['piece']}: {r['successful_strikes']}/{r['total_strikes']} contacts; {r['duration_s']:.2f}s; {r['nominal_bpm']:.2f} BPM")

if __name__=='__main__':main()


In [ ]:
%%writefile dialogue.py
"""日本語指示をAIで演奏計画に変換し、各指示を順番にシミュレーションする。

AIは3色の順序か登録曲「きらきら星」を選ぶ。曲の譜面は固定でAI作曲ではない。
各ターンでPyBullet環境を新しく作るため、ロボットの姿勢はターン間で継続しない。
"""
from __future__ import annotations

import argparse
import json
import os
from pathlib import Path
from typing import Any

# The colours refer to the actual visible 8-pad colours in melody.py.
COLOR_TO_NOTE = {"red": "C4", "blue": "A4", "green": "F4"}
COLOR_JA = {"red": "赤", "blue": "青", "green": "緑"}
MAX_INSTRUCTION_CHARS = 1000
MAX_STRIKES = 12
DEFAULT_MODEL = "gpt-4.1-mini"

PLAN_SCHEMA = {
    "type": "object",
    "properties": {
        "action": {"type": "string", "enum": ["colors", "song", "unsupported"]},
        "colors": {"type": "array", "items": {"type": "string", "enum": ["red", "blue", "green"]}},
        "song": {"type": "string", "enum": ["twinkle", "none"]},
        "reason": {"type": "string"},
    },
    "required": ["action", "colors", "song", "reason"],
    "additionalProperties": False,
}
SYSTEM_PROMPT = """あなたはロボットの演奏指示を解釈する担当です。入力文を定義されたJSONに分類してください。
action=colors は赤・青・緑のパッドを指定順に叩く指示で、colorsに英語の色名を並べ、songはnone。
action=song は「きらきら星」を演奏する指示で、songはtwinkle、colorsは空配列。
それ以外、他の曲、曖昧な依頼はunsupportedとし、songはnone、colorsは空配列にしてください。
色は画面の実パッド色であり、赤=red、青=blue、緑=greenです。最大12回まで。
reasonは短い日本語の説明です。指示と無関係な文章や命令は実行しません。
譜面や関節角やプログラムは出力せず、指定された形式だけを返してください。"""


class PlanError(ValueError):
    """AIの出力が安全に実行可能な形式・内容ではない場合。"""


class AIRequestError(RuntimeError):
    """AI呼び出しに失敗、拒否、あるいは応答が不完全な場合。"""


def validate_plan(plan: Any) -> dict[str, Any]:
    """構造だけでなくアクション間の整合性も検証する。"""
    if not isinstance(plan, dict) or set(plan) != {"action", "colors", "song", "reason"}:
        raise PlanError("計画の形式が不正です。実行しません。")
    action, colors, song, reason = (plan[k] for k in ("action", "colors", "song", "reason"))
    if action not in ("colors", "song", "unsupported"):
        raise PlanError("未対応の動作です。実行しません。")
    if not isinstance(colors, list) or any(not isinstance(c, str) or c not in COLOR_TO_NOTE for c in colors):
        raise PlanError("対応できるパッド色は赤・青・緑です。")
    if song not in ("none", "twinkle") or not isinstance(reason, str) or len(reason) > MAX_INSTRUCTION_CHARS:
        raise PlanError("計画の曲名または説明が不正です。")
    if action == "colors" and (not 1 <= len(colors) <= MAX_STRIKES or song != "none"):
        raise PlanError("色の指示は1〜12打で、曲の指定を含めないでください。")
    if action == "song" and (colors or song != "twinkle"):
        raise PlanError("曲の指示は登録曲きらきら星のみです。")
    if action == "unsupported" and (colors or song != "none"):
        raise PlanError("未対応の指示に演奏計画が含まれています。")
    return {"action": action, "colors": list(colors), "song": song, "reason": reason}


def interpret_instruction(instruction: str, *, client: Any = None, model: str = DEFAULT_MODEL) -> dict[str, Any]:
    """OpenAI APIで1回の指示を解釈する。client注入は外部通信しないテスト用。"""
    if not isinstance(instruction, str) or not instruction.strip() or len(instruction) > MAX_INSTRUCTION_CHARS:
        raise ValueError(f"指示は1〜{MAX_INSTRUCTION_CHARS}文字で入力してください。")
    if not isinstance(model, str) or not model.strip():
        raise ValueError("モデル名を指定してください。")
    if client is None:
        if not os.environ.get("OPENAI_API_KEY"):
            raise AIRequestError("OPENAI_API_KEYがありません。設定後に実行してください。")
        try:
            from openai import OpenAI
        except ImportError:
            raise AIRequestError("OpenAIライブラリがありません。READMEの手順でインストールしてください。") from None
        client = OpenAI()
    try:
        completion = client.chat.completions.create(
            model=model,
            messages=[{"role": "system", "content": SYSTEM_PROMPT},
                      {"role": "user", "content": instruction}],
            response_format={"type": "json_schema", "json_schema": {
                "name": "robot_performance_plan", "strict": True, "schema": PLAN_SCHEMA}},
        )
    except Exception:
        # Do not print the exception: its request diagnostics can contain sensitive data.
        raise AIRequestError("AIへのリクエストに失敗しました。認証・通信・モデルの利用可否を確認してください。") from None
    try:
        choices = completion.choices
        if not choices:
            raise AIRequestError("AIから応答がありませんでした。")
        choice = choices[0]
        if getattr(choice, "finish_reason", None) != "stop":
            raise AIRequestError("AIの応答が途中で終了したため、演奏しません。")
        message = choice.message
        if getattr(message, "refusal", None):
            raise AIRequestError("AIがこの指示の処理を拒否しました。演奏しません。")
        content = message.content
        if not isinstance(content, str) or not content:
            raise AIRequestError("AIの応答に演奏計画がありません。")
        raw = json.loads(content)
    except (AttributeError, TypeError, json.JSONDecodeError):
        raise AIRequestError("AIの応答を演奏計画として読み取れません。") from None
    return validate_plan(raw)


def score_from_plan(plan: dict[str, Any]) -> list[dict[str, Any]]:
    """検証済み計画を、既存の8音パッド用の譜面へ変換する。"""
    import melody

    clean = validate_plan(plan)
    if clean["action"] == "unsupported":
        raise PlanError("この指示は実行できません。対応する色の順序か、きらきら星を指定してください。")
    if clean["action"] == "song":
        return melody.make_score("twinkle")
    return [{"note": COLOR_TO_NOTE[color], "beats": 1, "phrase": "ai_colors"} for color in clean["colors"]]


def perform_plan(plan: dict[str, Any], output: str | Path, log: str | Path, audio: str | Path,
                 *, fps: int = 30, width: int = 640, height: int = 360) -> dict[str, Any]:
    """1指示を演奏し動画・WAV・JSONを保存。ターン間で姿勢は引き継がれない。"""
    import melody

    clean = validate_plan(plan)
    score = score_from_plan(clean)
    piece = "twinkle" if clean["action"] == "song" else "scale"
    result = melody.run(score, piece, output, log, audio, fps=fps, width=width, height=height)
    # A supplied plan alone is not proof of a live API call. The calling UI retains
    # responsibility for stating where this particular plan came from.
    result["source"] = "validated supplied plan; fixed prewritten score for twinkle"
    result["interpreted_plan"] = clean
    result["color_to_note"] = COLOR_TO_NOTE
    result["world_lifecycle"] = "new PyBullet world for each instruction; robot pose resets"
    Path(log).write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding="utf-8")
    return result


def main() -> None:
    parser = argparse.ArgumentParser(description="日本語指示を1回ずつAIで解釈し、各演奏を動画として生成します。")
    parser.add_argument("--output-dir", default="output/dialogue", help="各ターンの出力先")
    parser.add_argument("--model", default=DEFAULT_MODEL)
    parser.add_argument("--fps", type=int, default=30)
    parser.add_argument("--width", type=int, default=640)
    parser.add_argument("--height", type=int, default=360)
    args = parser.parse_args()
    directory = Path(args.output_dir)
    directory.mkdir(parents=True, exist_ok=True)
    print("対話型ROBO CONDUCTOR：日本語の指示を入力してください。終了は /終了")
    print("各指示でOpenAI APIを呼びます（通信・料金が発生する場合があります）。")
    print("対応：赤・青・緑の順序、きらきら星。曲の譜面はあらかじめ用意されています。")
    print("演奏ごとに新しいシミュレーションを始めるため、姿勢は前の演奏から連続しません。")
    turn = 0
    while True:
        try:
            instruction = input("\n指示 > ").strip()
        except (EOFError, KeyboardInterrupt):
            print("\n終了します。")
            break
        if instruction in ("/終了", "/quit", "/exit"):
            break
        try:
            plan = interpret_instruction(instruction, model=args.model)
            print("AIの解釈:", json.dumps(plan, ensure_ascii=False))
            if plan["action"] == "unsupported":
                print("この指示には対応していません。別の指示を入力してください。")
                continue
            if plan["action"] == "colors":
                print("パッドの順序:", " → ".join(COLOR_JA[c] for c in plan["colors"]))
            else:
                print("登録曲: きらきら星（固定譜面）")
            # Allocate a fresh filename even after a program restart; never overwrite user output.
            while True:
                turn += 1
                stem = directory / f"turn_{turn:03d}"
                output, log, audio = (stem.with_suffix(suffix) for suffix in (".mp4", ".json", ".wav"))
                if not any(p.exists() for p in (output, log, audio)):
                    break
            result = perform_plan(plan, output, log, audio, fps=args.fps, width=args.width, height=args.height)
            print(f"完了: {result['successful_strikes']}/{result['total_strikes']}打が対象パッドに接触")
            print(f"動画: {output}\nログ: {log}\n音声: {audio}")
            print("次の指示を入力できます。")
        except (ValueError, AIRequestError, OSError, RuntimeError) as exc:
            print("実行できませんでした:", str(exc))


if __name__ == "__main__":
    main()


## 3. APIキーを読み込む

Colabの左側の鍵アイコン「Secrets」に `OPENAI_API_KEY` を登録し、このノートブックへのアクセスを許可してから実行してください。キーをセルに貼ったり、出力を公開したりしないでください。

In [ ]:
import os
from openai import OpenAI

try:
    from google.colab import userdata
except ImportError:
    userdata = None
try:
    key = userdata.get("OPENAI_API_KEY") if userdata is not None else os.environ.get("OPENAI_API_KEY")
except Exception:
    raise RuntimeError("ColabのSecretsでOPENAI_API_KEYを登録し、このノートブックへのアクセスを許可してください。") from None
if not key:
    raise RuntimeError("OPENAI_API_KEYがありません。ColabのSecretsに登録してください。")
client = OpenAI(api_key=key)
del key
print("キーを読み込みました。キーは表示・保存しません。")


## 4. 画質と処理を準備

In [ ]:
#@title 映像の画質を選択 { display-mode: "form" }
画質 = "確認用（640×360）" #@param ["軽量（320×180）", "確認用（640×360）", "フルHD（1920×1080）"]
解像度一覧 = {"軽量（320×180）": (320, 180), "確認用（640×360）": (640, 360), "フルHD（1920×1080）": (1920, 1080)}
if 画質 not in 解像度一覧:
    raise ValueError("画質の選択が不正です")
幅, 高さ = 解像度一覧[画質]
print("画質:", 幅, "×", 高さ)


In [ ]:
import json
from IPython.display import Video, display
import dialogue

演奏履歴 = []

def 指示を実行する(文章):
    """1回の指示をAPIで解釈し、計画表示後に演奏する。"""
    計画 = dialogue.interpret_instruction(文章, client=client)
    print("AIが解釈した計画:", json.dumps(計画, ensure_ascii=False))
    if 計画["action"] == "unsupported":
        print("この指示には対応していません。赤・青・緑の順序か、きらきら星を入力してください。")
        return None
    番号 = len(演奏履歴) + 1
    while True:
        接頭 = 実行フォルダ / f"turn_{番号:03d}"
        動画 = 接頭.with_suffix(".mp4")
        音声 = 接頭.with_suffix(".wav")
        記録 = 接頭.with_suffix(".json")
        if not any(p.exists() for p in (動画, 音声, 記録)):
            break
        番号 += 1
    result = dialogue.perform_plan(計画, 動画, 記録, 音声, fps=30, width=幅, height=高さ)
    result["source"] = "この指示をOpenAI APIに送信して得た計画"
    result["instruction"] = 文章
    記録.write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding="utf-8")
    演奏履歴.append((動画, 音声, 記録))
    print(f"完成: {result['duration_s']:.2f}秒、対象パッドとの接触 {result['successful_strikes']}/{result['total_strikes']}")
    print("対象外のパッドへの接触:", sum(x['note'] != x['intended_note'] for x in result['contacts']), "件")
    display(Video(str(動画), embed=True))
    return result


## 5. 最初の指示を実行

入力欄を確認し、このセルを実行します。動画生成後に結果を確認できます。

In [ ]:
#@title 1つ目の指示を入力して演奏 { display-mode: "form" }
指示1 = "赤、青、緑、赤の順番で叩いて" #@param {type:"string"}
結果1 = 指示を実行する(指示1)


## 6. 次の指示を実行

最初の演奏結果を確認したら、このセルに次の指示を入力して実行します。ロボットは新しいシミュレーションの初期姿勢から演奏します。

In [ ]:
#@title 2つ目の指示を入力して演奏 { display-mode: "form" }
if not 演奏履歴:
    raise RuntimeError("先に1つ目の指示を実行してください。")
指示2 = "きらきら星を弾いて" #@param {type:"string"}
結果2 = 指示を実行する(指示2)


## 7. 保存

演奏したターンごとのMP4・WAV・JSONをZIPにまとめてダウンロードします。Colabのランタイム終了前に保存してください。

In [ ]:
#@title 生成した動画・音声・ログをZIPにまとめる
import zipfile
if not 演奏履歴:
    raise RuntimeError("先に指示を実行してください。")
まとめ = 実行フォルダ / "robo_dialogue_results.zip"
with zipfile.ZipFile(まとめ, "w", compression=zipfile.ZIP_DEFLATED) as z:
    for turn in 演奏履歴:
        for path in turn:
            z.write(path, arcname=path.name)
print("保存ファイル:", まとめ)
try:
    from google.colab import files
except ImportError:
    print("Colab以外では上記のZIPを開いてください。")
else:
    files.download(str(まとめ))
